# Mini-VLA Maze — portable validation

This notebook includes the Python source and the existing trained checkpoint in a compressed payload. No account key or dataset upload is needed. It evaluates the original 11×11 tasks and the new 25×25 arenas. The large arenas use learned visual heads + explicit A*, not the fixed-size neural action head.

Execution creates a provenance record, metrics, episode CSVs, a labelled animation and a downloadable result archive. It does not retrain the checkpoint.

In [ ]:
#@title Install dependencies and unpack the portable experiment
import base64,io,zipfile,sys,importlib.util,subprocess
from pathlib import Path
packages={'numpy':'numpy','torch':'torch','PIL':'Pillow','matplotlib':'matplotlib'}
missing=[p for m,p in packages.items() if importlib.util.find_spec(m) is None]
if missing: subprocess.check_call([sys.executable,'-m','pip','install',*missing])
ROOT=Path.cwd()/'mini_vla_runtime';ROOT.mkdir(exist_ok=True)
PAYLOAD='...'
with zipfile.ZipFile(io.BytesIO(base64.b64decode(PAYLOAD))) as z: z.extractall(ROOT)
sys.path.insert(0,str(ROOT/'src'))
print('Source and checkpoint ready:',ROOT)


## Runtime provenance
The evaluation functions run on CPU. GPU availability is recorded but is not evidence that these runs used a GPU.

In [ ]:
import torch,json,platform,hashlib,time,shutil
from datetime import datetime,timezone
RUN=Path.cwd()/'mini_vla_validation';RUN.mkdir(exist_ok=True)
torch.set_num_threads(min(4,torch.get_num_threads()))
provenance={'executed_at_utc':datetime.now(timezone.utc).isoformat(),'platform':platform.platform(),'python':platform.python_version(),'torch':torch.__version__,'cuda_available':torch.cuda.is_available(),'execution_device':'cpu','checkpoint_sha256':hashlib.sha256((ROOT/'weights.pt').read_bytes()).hexdigest()}
(RUN/'provenance.json').write_text(json.dumps(provenance,indent=2))
print(json.dumps(provenance,indent=2))

## Reference benchmark
Same checkpoint, same accepted scene seeds, 16 scenes per suite. Scores are compared through the saved records; local and Colab outputs remain separate.

In [ ]:
from mini_vla_maze.evaluate import run as evaluate
small=evaluate(ROOT/'weights.pt',RUN/'small',n=16)
print('REFERENCE_EVALUATION_COMPLETE')

## Larger held-out arenas
25×25 maps: repeated turns, nine connected rooms, and three timed passages. Six accepted scenes per arena. The exact tile renderer and known dynamics are retained; this is a constrained size-transfer test.

In [ ]:
from mini_vla_maze.large import run as evaluate_large
large=evaluate_large(ROOT/'weights.pt',RUN/'large',n=6)
print(json.dumps(large,indent=2))
print('LARGE_EVALUATION_COMPLETE')

## Watch a recorded rollout
The main view is styled for display. The raw camera image used by the neural model is shown separately.

In [ ]:
from mini_vla_maze.evaluate import load_model
from mini_vla_maze.large import make_arena,rollout
from mini_vla_maze.presentation import animation
from IPython.display import display,Image
model=load_model(ROOT/'weights.pt')
row,frames=rollout(model,make_arena(820002,'airlocks'),record=True)
animation(frames,row,RUN/'airlocks.gif')
print(row)
display(Image(filename=str(RUN/'airlocks.gif')))
shutil.copy2(ROOT/'weights.pt',RUN/'weights.pt')
archive=shutil.make_archive(str(RUN),'zip',RUN)
print('VALIDATION_COMPLETE',archive)

## Save the run
The archive contains both evaluations, seed manifests, provenance, the checkpoint and the animation. The notebook itself can also be downloaded with its execution outputs.

In [ ]:
try:
    from google.colab import files
    files.download(archive)
except ImportError:
    print('Local execution; result archive:',archive)